# Phase 16 — Split audit, group-signature definition and participant flow (Study 2)
**Resolves:** R2-10 (precise definition of the "identical feature-profile signature", stratification vs group
constraint), R2-15 (reproducibility: exact counts), R2 minor-1 (flow numbers must match text/figures), and the
Study 2 analogue of R2-1 (a single participant-flow table).

**What it does:** (1) documents the signature exactly as coded in `preprocessing.compute_group_id`;
(2) per dataset: partition sizes/percentages, prevalence per partition, #unique signatures, duplicated-profile
burden; (3) verifies that NO signature appears in more than one partition; (4) builds a raw -> analytic ->
train/cal/test flow table.
**Outputs:** `S2_participant_flow.csv`, `S2_split_audit.csv`, `S2_signature_definition.csv`.

In [1]:
import os, sys, json, warnings
import numpy as np
import pandas as pd
warnings.filterwarnings("ignore")
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), "..", "src")))
from revision_utils import *

ENV = get_env()          # paths + config (notebooks must be run from the notebooks/ folder)
SEED = ENV["SEED"]       # 42
B = ENV["N_BOOT"]        # 2000 bootstrap resamples (config.yaml)
print("Project root:", ENV["ROOT"], "| outputs ->", ENV["REV"])

Project root: C:\maternal-mental-health-risk | outputs -> C:\maternal-mental-health-risk\reports/tables\revision


In [2]:
from preprocessing import compute_group_id
audit_rows, sigdef_rows = [], []
for tag, d in DATASETS.items():
    S = load_split(tag, ENV)
    feats, lab = features_of(tag), d["label"]
    full = pd.concat([S["train"], S["cal"], S["test"]], ignore_index=True)
    sig = {k: compute_group_id(S[k][feats], feats) for k in ["train", "cal", "test"]}
    overall_prev = full[lab].mean()
    all_sig = pd.concat(sig.values(), ignore_index=True)
    for k, nm in [("train", "training"), ("cal", "calibration"), ("test", "test")]:
        df = S[k]
        counts = sig[k].value_counts()
        audit_rows.append(dict(
            dataset=tag, partition=nm, n=len(df), pct_of_analytic=100 * len(df) / len(full),
            n_positive=int(df[lab].sum()), prevalence=df[lab].mean(),
            prevalence_diff_vs_overall=df[lab].mean() - overall_prev,
            n_unique_signatures=int(counts.shape[0]),
            pct_rows_in_duplicated_profile=100 * float((sig[k].map(all_sig.value_counts()) > 1).mean()),
            max_rows_sharing_one_signature=int(counts.max())))
    # cross-partition overlap of signatures (must be 0 for the group-aware split)
    sets = {k: set(sig[k]) for k in sig}
    for a, b in [("train", "cal"), ("train", "test"), ("cal", "test")]:
        shared = sets[a] & sets[b]
        rows_b = int(sig[b].isin(shared).sum())
        audit_rows.append(dict(dataset=tag, partition=f"OVERLAP {a} & {b}", n=len(shared),
                               n_positive=rows_b, n_unique_signatures=len(shared)))
    # signature definition facts
    miss_any = full[feats].isna().any(axis=1).sum()
    for c in feats:
        sigdef_rows.append(dict(dataset=tag, variable=c, role="numeric" if c in d["numeric"] else "categorical",
                                n_distinct_values=int(full[c].nunique(dropna=True)),
                                n_missing=int(full[c].isna().sum()),
                                rounded_or_binned=False, missing_treated_as="literal string 'MISSING'"))
audit = pd.DataFrame(audit_rows)
save_table(audit, ENV, "S2_split_audit.csv")
save_table(pd.DataFrame(sigdef_rows), ENV, "S2_signature_definition.csv")
print(audit[audit.partition.str.startswith("OVERLAP")])
ov = audit[audit.partition.str.startswith("OVERLAP")]
bad = ov[ov["n"] > 0]
print("\nGROUP-AWARE SPLIT CHECK:", "PASSED - no signature crosses partitions in any dataset" if bad.empty else
      "ATTENTION - identical signatures cross partitions for:\n" + bad[["dataset", "partition", "n", "n_positive"]].to_string())

Saved: C:\maternal-mental-health-risk\reports/tables\revision\S2_split_audit.csv  (24 rows)
Saved: C:\maternal-mental-health-risk\reports/tables\revision\S2_signature_definition.csv  (68 rows)
             dataset             partition  n  pct_of_analytic  n_positive  \
3           pakistan   OVERLAP train & cal  0              NaN           0   
4           pakistan  OVERLAP train & test  0              NaN           0   
5           pakistan    OVERLAP cal & test  0              NaN           0   
9   arab5_depression   OVERLAP train & cal  0              NaN           0   
10  arab5_depression  OVERLAP train & test  0              NaN           0   
11  arab5_depression    OVERLAP cal & test  0              NaN           0   
15     arab5_anxiety   OVERLAP train & cal  0              NaN           0   
16     arab5_anxiety  OVERLAP train & test  0              NaN           0   
17     arab5_anxiety    OVERLAP cal & test  0              NaN           0   
21            uganda   OVER

**Reading the overlap rows:** `n` = number of signatures present in both partitions and `n_positive` here holds the
number of rows in the later partition carrying such a signature. For a correct group-aware split both are 0.
**Uganda note:** rare districts (<30 training rows) were collapsed to "Other" AFTER the split (notebook 05c).
The overlap above is computed on the collapsed data actually used for modelling; any non-zero Uganda overlap
must be reported (and the Uganda sensitivity re-split described in the revision plan run).

In [3]:
# Raw -> analytic -> partitions flow table, from the interim files written by notebook 03/04
flow = []
def _ip(name): return os.path.join(ENV["INTERIM"], name)
def _n(tag, k): return len(load_split(tag, ENV)[k])

if os.path.exists(_ip("pakistan_interim.csv")):
    n_raw = len(pd.read_csv(_ip("pakistan_interim.csv")))
    flow.append(dict(dataset="pakistan", step="Raw records (Lahore, Gujranwala)", n=n_raw))
    flow.append(dict(dataset="pakistan", step="Excluded: none (all rows retained; PHQ-9 items/sum removed as predictors, not rows)", n=0))
if os.path.exists(_ip("arab5_interim.csv")):
    ar = pd.read_csv(_ip("arab5_interim.csv"))
    flow.append(dict(dataset="arab5", step="Raw records (5 countries)", n=len(ar)))
    flow.append(dict(dataset="arab5_depression", step="Excluded: incomplete PHQ-9 items (default-imputed to lowest category)", n=int((~ar["dep_items_complete"]).sum())))
    flow.append(dict(dataset="arab5_depression", step="Primary depression analytic sample", n=int(ar["dep_items_complete"].sum())))
    flow.append(dict(dataset="arab5_anxiety", step="Excluded: incomplete GAD-7 items", n=int((~ar["anx_items_complete"]).sum())))
    flow.append(dict(dataset="arab5_anxiety", step="Primary anxiety analytic sample", n=int(ar["anx_items_complete"].sum())))
if os.path.exists(_ip("uganda_interim.csv")):
    flow.append(dict(dataset="uganda", step="Raw records (perinatal anxiety screening programme)", n=len(pd.read_csv(_ip("uganda_interim.csv")))))
for tag in DATASETS:
    S = load_split(tag, ENV); N = sum(len(S[k]) for k in ["train", "cal", "test"])
    flow.append(dict(dataset=tag, step="Analytic sample (train+calibration+test)", n=N))
    for k, nm in [("train", "Training (~60%)"), ("cal", "Calibration (~20%)"), ("test", "Test (~20%)")]:
        flow.append(dict(dataset=tag, step=nm, n=len(S[k]), pct=round(100 * len(S[k]) / N, 1),
                         prevalence=round(float(S[k][DATASETS[tag]["label"]].mean()), 4)))
ext = os.path.join(ENV["EXTERNAL"])
for nm, fname in [("pakistan_harmonized", "pakistan_harmonized.csv"), ("arab5_harmonized", "arab5_harmonized.csv")]:
    p = os.path.join(ext, fname)
    if os.path.exists(p):
        flow.append(dict(dataset=nm, step="Cross-national analytic sample after dropping rows with missing harmonised features", n=len(pd.read_csv(p))))
save_table(pd.DataFrame(flow), ENV, "S2_participant_flow.csv")
pd.DataFrame(flow)

Saved: C:\maternal-mental-health-risk\reports/tables\revision\S2_participant_flow.csv  (26 rows)


,dataset,step,n,pct,prevalence
0,pakistan,"Raw records (Lahore, Gujranwala)",14008,NaN,NaN
1,pakistan,Excluded: none (all rows retained; PHQ-9 items...,0,NaN,NaN
2,arab5,Raw records (5 countries),3569,NaN,NaN
3,arab5_depression,Excluded: incomplete PHQ-9 items (default-impu...,2070,NaN,NaN
4,arab5_depression,Primary depression analytic sample,1499,NaN,NaN
5,arab5_anxiety,Excluded: incomplete GAD-7 items,2623,NaN,NaN
6,arab5_anxiety,Primary anxiety analytic sample,946,NaN,NaN
7,uganda,Raw records (perinatal anxiety screening progr...,14325,NaN,NaN
8,pakistan,Analytic sample (train+calibration+test),14008,NaN,NaN
9,pakistan,Training (~60%),8402,60.0,0.6531
